# บทที่ 11 - โปรโตคอลจากเอเย่นต์ถึงเอเย่นต์ (A2A)


## การตั้งค่า


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## โปรโตคอล A2A คืออะไร?

**โปรโตคอล Agent-to-Agent (A2A)** คือมาตรฐานเปิดที่ช่วยให้เอเย่นต์ AI สามารถสื่อสาร,
ค้นหากันและกัน และร่วมมือกัน — แม้ว่าพวกเขาจะถูกสร้างขึ้นบนกรอบงานที่แตกต่างกันหรือตั้งอยู่
บนบริการที่ต่างกัน

แนวคิดหลัก:

- **การค้นหา** – เอเย่นต์เผยแพร่ *บัตรเอเย่นต์* ที่อธิบายความสามารถของตน ทำให้
  ง่ายต่อการที่เอเย่นต์อื่น (หรือผู้ประสานงาน) จะค้นหาผู้เชี่ยวชาญที่เหมาะสมสำหรับงาน
- **การส่งข้อความ** – เอเย่นต์แลกเปลี่ยนข้อความที่มีโครงสร้างผ่านโปรโตคอลร่วม เพื่อคำขอ
  จากเอเย่นต์หนึ่งจะถูกเข้าใจและดำเนินการโดยเอเย่นต์อีกตัวหนึ่งโดยไม่คำนึงถึงการ
  ดำเนินการภายในของมัน
- **วงจรชีวิตของงาน** – A2A กำหนดสถานะ เช่น *ส่งแล้ว*, *กำลังทำงาน*, *เสร็จสมบูรณ์*, และ
  *ล้มเหลว* ช่วยให้ผู้ประสานงานมองเห็นภาพรวมของความก้าวหน้าในงานที่มอบหมาย

ในบทเรียนนี้ เราจะจำลองการร่วมมือสไตล์ A2A โดยเชื่อมต่อเอเย่นต์ท่องเที่ยวเฉพาะทางสามตัว
เข้าในเวิร์กโฟลว์ที่แต่ละเอเย่นต์มีส่วนร่วมด้วยความเชี่ยวชาญของตนและส่งผลลัพธ์ให้กับตัวถัดไป


## การสร้างตัวแทนท่องเที่ยวเฉพาะทาง


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## การทำงานร่วมกันของหลายตัวแทนผ่านเวิร์กโฟลว์

เราเชื่อมต่อสามตัวแทนเข้ากับเวิร์กโฟลว์แบบเรียงลำดับซึ่งสะท้อนการส่งข้อความ A2A:

1. **CurrencyExchangeAgent** รับคำขอจากผู้ใช้และให้คำแนะนำเกี่ยวกับสกุลเงิน
2. **ActivityPlannerAgent** รับบริบทที่ได้รับการเติมเต็มและเพิ่มคำแนะนำกิจกรรม
3. **TravelManagerAgent** สังเคราะห์ข้อมูลทั้งสองเป็นสรุปการเดินทางขั้นสุดท้าย


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## ความเข้าใจ A2A ในการใช้งานจริง

ในสภาพแวดล้อมการใช้งานจริง โปรโตคอล A2A เปิดใช้งานสถานการณ์ข้ามบริการที่ทรงพลัง:

| ความสามารถ | คำอธิบาย |
|---|---|
| **การทำงานร่วมกันข้ามเฟรมเวิร์ก** | ตัวแทนที่สร้างด้วยเฟรมเวิร์กหนึ่งสามารถมอบหมายงานให้กับตัวแทนที่สร้างด้วยเฟรมเวิร์ก A2A ที่รองรับอื่น ๆ ได้ ช่วยให้สามารถทำงานร่วมกันข้ามองค์กรได้อย่างแท้จริง |
| **ขอบเขตของบริการ** | ตัวแทนสามารถอยู่ในไมโครเซอร์วิส แถบภูมิภาคคลาวด์ หรือแม้แต่ต่างองค์กรกัน และยังสามารถทำงานร่วมกันได้อย่างราบรื่น |
| **การค้นพบแบบไดนามิก** | ตัวควบคุมสามารถสืบค้น Agent Card registry ในเวลาทำงานเพื่อค้นหาผู้เชี่ยวชาญที่เหมาะสมที่สุดสำหรับงานย่อยที่กำหนด |
| **การสตรีมและการแจ้งเตือนแบบพุช** | A2A รองรับ Server-Sent Events (SSE) สำหรับอัปเดตความคืบหน้าแบบเรียลไทม์และการแจ้งเตือนแบบพุชสำหรับงานที่ใช้เวลานาน |

เวิร์กโฟลว์ที่เราสร้างไว้ข้างต้นเป็นเวอร์ชันที่เรียบง่ายภายในกระบวนการของรูปแบบนี้ ในการ
ปล่อยใช้งานจริง ตัวแทนแต่ละตัวจะเปิดเผย HTTP endpoint เผยแพร่ Agent Card และสื่อสาร
ผ่านโปรโตคอล A2A JSON-RPC


## สรุป

ในบทเรียนนี้คุณได้เรียนรู้:

1. **โปรโตคอล A2A คืออะไร** — มาตรฐานเปิดสำหรับการค้นหา ส่งข้อความระหว่างเอเจนต์
   และการจัดการงาน
2. **วิธีสร้างเอเจนต์เฉพาะทาง** — เอเจนต์แลกเปลี่ยนสกุลเงิน, เอเจนต์วางแผนกิจกรรม,
   และผู้จัดการเดินทางแบบออร์เคสตราเตอร์
3. **วิธีเชื่อมต่อเอเจนต์ในเวิร์กโฟลว์** — ใช้ `WorkflowBuilder` เพื่อสร้างแบบจำลองการส่งข้อความต่อเนื่องระหว่างเอเจนต์

4. **วิธีที่ A2A ทำงานในสภาพแวดล้อมจริง** — เปิดใช้งานความร่วมมือข้ามเฟรมเวิร์กและบริการ
   ด้วยการค้นหาแบบไดนามิกและการอัปเดตแบบสตรีม


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ปฏิเสธความรับผิดชอบ**:
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษา AI [Co-op Translator](https://github.com/Azure/co-op-translator) ขณะที่เราพยายามให้ความถูกต้อง โปรดทราบว่าการแปลโดยอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่ถูกต้อง เอกสารต้นฉบับในภาษาต้นทางควรถูกพิจารณาเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่สำคัญ แนะนำให้ใช้การแปลโดยมนุษย์มืออาชีพ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความที่ผิดพลาดที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
